In [10]:
SYSTEM_PROMPT="""ROLE & OBJECTIVE

You are a professional brochure layout engine auditor.

you are given a brochure pdf your task is to audit the brochure and find out if the brochure is broken in structure or not. A brochure is considered broken in structure if the content is not organized in a logical and visually appealing manner, making it difficult for readers to understand the information being presented.

AUDIT CRITERIA
1. Layout and Design: The brochure should have a clean and organized layout with a consistent design theme. The use of colors, fonts, and images should be harmonious and visually appealing.
2. No image or text overlap: The brochure should not have any overlapping images or text, as this can make it difficult to read and understand the content.
3. Proper alignment: The text and images should be properly aligned, creating a balanced and professional appearance.
4. No empty pages: The brochure should not contain any empty pages, as this can give the impression of a lack of attention to detail.
5. No broken images: All images in the brochure should be intact and properly displayed, without any signs of corruption or distortion.
6. No broken characters: The text in the brochure should be free from any broken characters or symbols, ensuring that the content is easily readable.

RESPONSE FORMAT
Your response should be in JSON format with the following structure:

{
  "is_broken": boolean, // true if the brochure is broken in structure, false otherwise
  "issues": [ // list of identified issues in the brochure
    {
      "issue_type": string, // type of issue (e.g., "Layout and Design", "Image Overlap", etc.)
      "description": string // detailed description of the issue
    },
    ...
  ]
}
"""

In [11]:
import os
import json
import logging
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from google import genai
from google.genai import types

# =======================
# ENV & CONFIG
# =======================

load_dotenv()

CSV_PATH = "xids.xlsx"
OUTPUT_CSV = "brochure_validation_output.csv"
BROCHURE_FOLDER = "Brochures_remaining_2/Brochures"
MODEL_ID = "gemini-3-flash-preview"

MAX_WORKERS = 50        # safer for API limits
MAX_RETRIES = 3
BACKOFF_FACTOR = 2       # exponential backoff multiplier

# =======================
# LOGGING CONFIG
# =======================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

log_lock = Lock()

# =======================
# GEMINI CLIENT FACTORY
# =======================

def create_client():
    return genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# =======================
# RETRY DECORATOR
# =======================

def retry_with_backoff(func):
    def wrapper(*args, **kwargs):
        delay = 1
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                if attempt == MAX_RETRIES:
                    raise
                logging.warning(
                    f"Retry {attempt}/{MAX_RETRIES} after error: {e}"
                )
                time.sleep(delay)
                delay *= BACKOFF_FACTOR
    return wrapper

# =======================
# CORE WORKER FUNCTION
# =======================

@retry_with_backoff
def process_project(idx, row_dict):
    project_id = str(row_dict.get("XID", f"row_{idx}")).strip()

    brochure_path = os.path.join(BROCHURE_FOLDER, f"{project_id}.pdf")

    if not os.path.exists(brochure_path):
        return {
            "idx": idx,
            "project_id": project_id,
            "status": "brochure_missing",
            "is_broken": None,
            "issues": "Brochure file not found"
        }

    client = create_client()

    file = client.files.upload(file=brochure_path)

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[file],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0,
            system_instruction=SYSTEM_PROMPT,
        ),
    )

    if not hasattr(response, "text") or not response.text.strip():
        return {
            "idx": idx,
            "project_id": project_id,
            "status": "empty_response",
            "is_broken": None,
            "issues": "Empty model response"
        }

    try:
        data = json.loads(response.text)
    except json.JSONDecodeError:
        return {
            "idx": idx,
            "project_id": project_id,
            "status": "invalid_json",
            "is_broken": None,
            "issues": "Invalid JSON returned"
        }

    if data.get("is_broken"):
        issues = data.get("issues", [])
        issue_descriptions = "; ".join(
            f"{issue.get('issue_type')}: {issue.get('description')}"
            for issue in issues
        )

        print(f"{project_id} → Brochure is broken: {issue_descriptions}")

        return {
            "idx": idx,
            "project_id": project_id,
            "status": "completed",
            "is_broken": "Yes",
            "issues": issue_descriptions,
        }  
    else:
        print(f"{project_id} → Brochure is not broken")
        return {
            "idx": idx,
            "project_id": project_id,
            "status": "completed",
            "is_broken": "No",
            "issues": "",
        } 

# =======================
# MAIN EXECUTION
# =======================

def main():
    df = pd.read_excel(CSV_PATH)

    # Ensure output columns exist
    if "Is Brochure Broken" not in df.columns:
        df["Is Brochure Broken"] = None
    if "Issues" not in df.columns:
        df["Issues"] = None

    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(process_project, idx, row.to_dict())
            for idx, row in df.iterrows()
        ]

        for future in tqdm(as_completed(futures), total=len(futures)):
            try:
                result = future.result()
                results.append(result)
                logging.info(f"{result['project_id']} → {result['status']}")
            except Exception as e:
                logging.error(f"Unhandled failure: {e}")

    # =======================
    # APPLY RESULTS TO DF
    # =======================

    for result in results:
        idx = result["idx"]
        df.loc[idx, "Is Brochure Broken"] = result["is_broken"]
        df.loc[idx, "Issues"] = result["issues"]

    # =======================
    # SAVE OUTPUT
    # =======================

    df.to_csv(OUTPUT_CSV, index=False)
    logging.info(f"Processing completed. Output saved to {OUTPUT_CSV}")

# =======================
# ENTRYPOINT
# =======================

if __name__ == "__main__":
    main()


  0%|          | 0/190 [00:00<?, ?it/s]2026-03-05 18:58:58,425 - INFO - 461039 → brochure_missing
2026-03-05 18:58:58,443 - INFO - 252013 → brochure_missing
2026-03-05 18:58:58,445 - INFO - 316319 → brochure_missing
2026-03-05 18:58:58,463 - INFO - 439996 → brochure_missing
2026-03-05 18:58:58,463 - INFO - 460425 → brochure_missing
2026-03-05 18:58:58,463 - INFO - 459938 → brochure_missing
2026-03-05 18:58:58,463 - INFO - 413291 → brochure_missing
2026-03-05 18:58:58,471 - INFO - 27994 → brochure_missing
2026-03-05 18:58:58,476 - INFO - 461132 → brochure_missing
2026-03-05 18:58:58,479 - INFO - 461353 → brochure_missing
2026-03-05 18:58:58,479 - INFO - 460204 → brochure_missing
2026-03-05 18:58:58,485 - INFO - 453471 → brochure_missing
2026-03-05 18:58:58,486 - INFO - 460669 → brochure_missing
2026-03-05 18:58:58,486 - INFO - 459500 → brochure_missing
2026-03-05 18:58:58,486 - INFO - 458270 → brochure_missing
2026-03-05 18:58:58,489 - INFO - 422861 → brochure_missing
2026-03-05 18:58:5

460444 → Brochure is broken: Image Overlap: On page 2, the text 'ARCHITECTURAL EXCELLENCE' overlaps with the circular graphic element, which compromises the visual clarity and professional design of the brochure.


2026-03-05 19:00:22,428 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWx-XfP2AbKBla9skXG2k-LjkeTKo1pq_OLLYh6kzB3FmneSohg5YXSO5bHpM9YpmlJdfu4PiTQ-6SMSg9uIAZ66beL7eHKP5PMkjHhuyMI&upload_protocol=resumable "HTTP/1.1 200 OK"
2026-03-05 19:00:22,440 - INFO - AFC is enabled with max remote calls: 10.
2026-03-05 19:00:25,005 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"
2026-03-05 19:00:25,021 - INFO - 414180 → completed
 87%|████████▋ | 165/190 [01:26<00:21,  1.19it/s]

414180 → Brochure is broken: Text Overlap: On page 1, the text 'Western Mumbai' overlaps directly with '32 FLOORS' in the bottom center section, creating a significant readability issue.; Layout and Design: On page 4, the text 'SHAGUN RESIDENC' in the Location Map is cut off at the edge of the graphic, missing the final letter 'Y'.


2026-03-05 19:00:26,659 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"
2026-03-05 19:00:26,678 - INFO - 405693 → completed


...

462505 → Brochure is broken: Layout and Design: The brochure contains a significant logical inconsistency in its content organization. On pages 1 and 2, the project is simultaneously described as 'READY TO MOVE' and having a 'COMPLETION: FEB 2026' date. This contradictory information makes it difficult for the reader to understand the actual status of the development.; Layout and Design: On page 4, there is a layout error where a section header is missing. While a gold decorative line is present to denote a new section, the title text (identified as 'INVESTMENT' in the OCR data) is not visible in the visual layout, creating an incomplete and unprofessional appearance.


2026-03-05 19:00:49,952 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"
2026-03-05 19:00:49,952 - INFO - 426696 → completed


426696 → Brochure is broken: Layout and Design: On page 2, the 'Area' in the Quick Facts section is displayed with nine decimal places (0.049964708 acres), which is visually unappealing and lacks professional formatting.; Image or Text Overlap: On page 4, the OCR identifies text such as 'Investment' and 'Palakkad Connectivity Hub' that is not visible in the screenshot, suggesting these elements are hidden behind other shapes or layers (e.g., the blue pricing box or footer elements).


2026-03-05 19:00:59,372 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"
2026-03-05 19:00:59,388 - INFO - 450749 → completed
100%|██████████| 190/190 [02:00<00:00,  1.57it/s]
2026-03-05 19:00:59,500 - INFO - Processing completed. Output saved to brochure_validation_output.csv


450749 → Brochure is broken: Layout and Design: The brochure contains inconsistent text formatting. On page 2, the status is displayed as a technical string 'READY_TO_MOVE' with underscores, whereas on page 3, it is correctly formatted as 'Ready to Move'.; Layout and Design: There is a logical inconsistency in the 'Quick Facts' section on page 2; the project status is listed as 'READY_TO_MOVE', yet the completion date is set in the future (Aug, 2025).; Layout and Design: The page numbering is inconsistent across the document. Page 1 lacks a page number in the bottom-right corner, while pages 2 and 3 include '02' and '03' respectively in that position.
